# Synchronized patch sampling

This notebook demonstrates MedImageFlow's patch workflow: multiple aligned fields share one sampled center and are cropped with the same geometry.

## Learning objectives

- Create aligned synthetic image, label, and ROI arrays.
- Configure `PatchDataset` with `RandomPatchSampler`.
- Use an ROI mask to guide random center sampling.
- Inspect patch shape, center, and location.
- Visualize both the source region and cropped patches.
- Understand constant padding for out-of-bounds centers.

## Requirements

Install the notebook dependencies from the repository root:

```bash
python -m pip install -e ".[notebooks]"
```

This notebook uses `.npy` files, so no DICOM or NIfTI data is required.

## Example data

Synthetic arrays are written to `examples/data/generated/patch_sampling/`.

## 1. Imports

In [ ]:
from collections.abc import Iterable
from pathlib import Path

import matplotlib.patches as patches
import matplotlib.pyplot as plt
import numpy as np

from medimageflow.data import PatchDataset, RandomPatchSampler, Sample

## 2. Configuration

In [ ]:
def find_repo_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for path in (start, *start.parents):
        if (path / "pyproject.toml").exists() and (path / "src" / "medimageflow").exists():
            return path
    raise RuntimeError("Could not find the MedImageFlow repository root.")


REPO_ROOT = find_repo_root()
DATA_DIR = REPO_ROOT / "examples" / "data" / "generated" / "patch_sampling"
DATA_DIR.mkdir(parents=True, exist_ok=True)
print(DATA_DIR)

## 3. Load or create data

The image, label, and ROI have the same spatial shape. This alignment is required because MedImageFlow does not register or resample modalities automatically.

In [ ]:
rows, cols = np.indices((64, 64))
image = (0.5 * rows + cols).astype(np.float32)
image += 40 * np.exp(-(((rows - 36) ** 2 + (cols - 28) ** 2) / 160))
label = (((rows - 36) ** 2 + (cols - 28) ** 2) < 9 ** 2).astype(np.uint8)
roi = (((rows - 36) ** 2 + (cols - 28) ** 2) < 16 ** 2).astype(np.uint8)

image_path = DATA_DIR / "image.npy"
label_path = DATA_DIR / "label.npy"
roi_path = DATA_DIR / "roi.npy"
np.save(image_path, image)
np.save(label_path, label)
np.save(roi_path, roi)

sample = Sample(
    paths={"image": image_path, "label": label_path, "roi": roi_path},
    id="synthetic-case",
)

print(image.shape, label.shape, roi.shape)

## 4. Run the MedImageFlow workflow

`RandomPatchSampler` selects one center. `PatchDataset` then crops all `patch_keys` around that center. With `center_mask_key="roi"`, centers are sampled from ROI foreground voxels.

In [ ]:
sampler = RandomPatchSampler((24, 24), seed=7)

patch_dataset = PatchDataset(
    [sample],
    sampler=sampler,
    spatial_dims=2,
    patch_keys=("image", "label", "roi"),
    reference_key="image",
    center_mask_key="roi",
)

patch = patch_dataset[0]
print("center:", patch["patch_center"])
print("location:", patch["patch_location"])
print("image patch:", patch["image"].shape)
print("label patch:", patch["label"].shape)
print("roi patch:", patch["roi"].shape)

A fixed seed makes the same sample index reproducible.

In [ ]:
first_center = patch_dataset[0]["patch_center"]
second_center = patch_dataset[0]["patch_center"]
np.testing.assert_array_equal(first_center, second_center)
print(first_center)

## 5. Inspect synchronized crops

The image, label, and ROI patches share the same `patch_center` and `patch_location`, so their voxel coordinates remain aligned.

In [ ]:
start_row, start_col = patch["patch_location"]
height, width = patch["image"].shape

print("source start:", (int(start_row), int(start_col)))
print("source stop:", (int(start_row + height), int(start_col + width)))
print("label foreground voxels in patch:", int(patch["label"].sum()))

## 6. Visualize results

In [ ]:
figure, axes = plt.subplots(1, 3, figsize=(12, 4))

axes[0].imshow(image, cmap="gray")
axes[0].imshow(label, cmap="Reds", alpha=0.35)
rectangle = patches.Rectangle(
    (start_col, start_row),
    width,
    height,
    fill=False,
    edgecolor="cyan",
    linewidth=2,
)
axes[0].add_patch(rectangle)
axes[0].set_title("source + patch box")

axes[1].imshow(patch["image"], cmap="gray")
axes[1].set_title("image patch")

axes[2].imshow(patch["label"], cmap="gray")
axes[2].set_title("label patch")

for axis in axes:
    axis.axis("off")
figure.tight_layout()
figure

## 7. Padding behavior

Random sampling can request centers near or outside the image boundary when `center_range` is configured. `PatchDataset` pads the crop to the requested shape. The short custom sampler below is only used to make the padding example deterministic.

In [ ]:
class FixedCenterSampler:
    def __init__(self, patch_size: tuple[int, int], center: tuple[int, int]) -> None:
        self.patch_size = patch_size
        self.center = center

    def centers(
        self,
        shape: tuple[int, int],
        count: int,
        *,
        center_mask=None,
        seed_offset: int = 0,
    ) -> Iterable[tuple[int, int]]:
        del shape, center_mask, seed_offset
        for _ in range(count):
            yield self.center


padding_dataset = PatchDataset(
    [sample],
    sampler=FixedCenterSampler((24, 24), center=(-4, 8)),
    spatial_dims=2,
    patch_keys=("image", "label"),
    reference_key="image",
    center_mask_key=None,
    padding_mode="constant",
    padding_value={"image": 0, "label": 0},
)

padded_patch = padding_dataset[0]
print("center:", padded_patch["patch_center"])
print("location:", padded_patch["patch_location"])
print("shape:", padded_patch["image"].shape)

In [ ]:
figure, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(padded_patch["image"], cmap="gray")
axes[0].set_title("padded image patch")
axes[1].imshow(padded_patch["label"], cmap="gray")
axes[1].set_title("padded label patch")
for axis in axes:
    axis.axis("off")
figure.tight_layout()
figure

## 8. Next steps

Patch sampling is useful when full 3D volumes are too large for model training or when you want to oversample foreground regions. MedImageFlow keeps the crop synchronized across fields, but it does not perform registration, resampling, or orientation normalization; prepare aligned inputs before building the dataset.